# Pydantic: Core Concepts

Pydantic is a Python library for defining **schemas** and **validating, parsing, and serializing** structured data using type hints.

```text
                         Pydantic
                            │
       ┌────────────────────┼────────────────────┐
       ↓                    ↓                    ↓
   Type Hints           Validation         Serialization
       │                    │                    │
       ↓                    ↓                    ↓
 Define schema         Check data          Python / JSON
                            │
                    ┌───────┴───────┐
                    ↓               ↓
                 Lax Mode       Strict Mode
                    │               │
               Coerce types     No coercion
                    │
                    ↓
             Custom Validators
                    │
                    ↓
             Field Constraints
                    │
                    ↓
               JSON Schema
```

### 🧠 Core Mental Model

- `Pydantic = Schema + Validation + Parsing + Serialization`

- **`pydantic-core`** is the Rust-based core responsible for high-performance validation and parsing.

- Pydantic is widely used in frameworks and packages such as **FastAPI, LangChain, and the OpenAI Python SDK**.

### 🔑 Key Points

* Invalid data → `ValidationError`
* Pydantic v2 custom validators → `@field_validator`
* Lax mode → allows compatible type coercion
* Strict mode → prevents type coercion
* `Field()` → defines constraints and metadata
* JSON Schema → machine-readable representation of the model

**📦 Install Pydantic:** `!python -m pip install pydantic`

**📚 Official Documentation:** [Pydantic Validation Guide](https://pydantic.dev/docs/validation/latest/get-started/)

In [2]:
# 1. model : datatype(primitive,collections), field definitions, validations
# 2. Print the JSON schema for the model

import json
from pydantic import (
    BaseModel, Field,
    StrictInt, StrictStr, StrictFloat, StrictBool,
    field_validator
)
from datetime import date, datetime
from decimal import Decimal
from uuid import UUID


class Address(BaseModel):

    city: str = Field(..., description="Customer's city")
    country: str = Field("India", description="Customer's country")


class Customer(BaseModel):

    # INTEGER
    age: int = Field(..., ge=18, le=100, description="Customer's age")
    age_optional: int | None = Field(None, ge=18, le=100, description="Optional age")
    strict_age: StrictInt = Field(..., ge=18, le=100, description="Strict customer age")
    strict_age_optional: StrictInt | None = Field(None, ge=18, le=100, description="Optional strict age")

    # STRING
    name: str = Field(..., min_length=2, max_length=50, description="Customer's name")
    name_optional: str | None = Field(None, min_length=2, max_length=50, description="Optional name")
    strict_name: StrictStr = Field(..., min_length=2, max_length=50, description="Strict customer name")
    strict_name_optional: StrictStr | None = Field(None, min_length=2, max_length=50, description="Optional strict name")

    # FLOAT
    salary: float = Field(..., gt=0, description="Customer's salary")
    salary_optional: float | None = Field(None, gt=0, description="Optional salary")
    strict_salary: StrictFloat = Field(..., gt=0, description="Strict customer salary")
    strict_salary_optional: StrictFloat | None = Field(None, gt=0, description="Optional strict salary")

    # BOOLEAN
    active: bool = Field(..., description="Customer active status")
    active_optional: bool | None = Field(None, description="Optional active status")
    strict_active: StrictBool = Field(..., description="Strict active status")
    strict_active_optional: StrictBool | None = Field(None, description="Optional strict active status")

    # OTHER TYPES
    dob: date = Field(..., description="Date of birth")
    created_at: datetime = Field(..., description="Account creation time")
    balance: Decimal = Field(..., gt=0, description="Account balance")
    customer_id: UUID = Field(..., description="Unique customer ID")

    # COLLECTIONS
    skills: list[str] = Field(default_factory=list, description="Customer skills")
    roles: set[str] = Field(default_factory=set, description="Unique customer roles")
    location: tuple[str, str] = Field(..., description="City and country")
    metadata: dict[str, str] = Field(default_factory=dict, description="Additional data")

    # NESTED MODEL
    address: Address = Field(..., description="Customer address")


    # FIELD VALIDATORS
    @field_validator("age")
    @classmethod
    def validate_age(cls, value):
        if value > 100:
            raise ValueError("Age cannot exceed 100")
        return value


    @field_validator("name")
    @classmethod
    def validate_name(cls, value):
        if not value.replace(" ", "").isalpha():
            raise ValueError("Name must contain only letters")
        return value


    @field_validator("salary")
    @classmethod
    def validate_salary(cls, value):
        if value > 10_000_000:
            raise ValueError("Salary cannot exceed 10,000,000")
        return value


    @field_validator("active")
    @classmethod
    def validate_active(cls, value):
        if not isinstance(value, bool):
            raise ValueError("Active must be True or False")
        return value


    @field_validator("dob")
    @classmethod
    def validate_dob(cls, value):
        if value > date.today():
            raise ValueError("Date of birth cannot be in the future")
        return value


    @field_validator("skills")
    @classmethod
    def validate_skills(cls, value):
        if len(value) > 10:
            raise ValueError("Maximum 10 skills allowed")
        return value



print(json.dumps(Customer.model_json_schema(), indent=2))

# str= Tries to coerce the value to a string, but will raise an error if it cannot be coerced.
# str |None= Tries to coerce the value to a string, but will raise an error if it cannot be coerced. If the value is None, it will be accepted.
# strictStr= Requires the value to be a string, and will raise an error if it is not. It does not allow coercion.
# strictStr | None= Requires the value to be a string, and will raise an error if it is not. It does not allow coercion. If the value is None, it will be accepted.

{
  "$defs": {
    "Address": {
      "properties": {
        "city": {
          "description": "Customer's city",
          "title": "City",
          "type": "string"
        },
        "country": {
          "default": "India",
          "description": "Customer's country",
          "title": "Country",
          "type": "string"
        }
      },
      "required": [
        "city"
      ],
      "title": "Address",
      "type": "object"
    }
  },
  "properties": {
    "age": {
      "description": "Customer's age",
      "maximum": 100,
      "minimum": 18,
      "title": "Age",
      "type": "integer"
    },
    "age_optional": {
      "anyOf": [
        {
          "maximum": 100,
          "minimum": 18,
          "type": "integer"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Optional age",
      "title": "Age Optional"
    },
    "strict_age": {
      "description": "Strict customer age",
      "maximum": 100,